# **Phase 3: Feature Engineering**

- Sequence features (order, recency, frequency) - ahi esta malparidos
- Job features (text similarity with NLP) - ahi esta malparidos
- Session features (behavior patterns) - ahi esta malparidos


## **Setup and Data Loading**

In [43]:
import sys
sys.path.append('..')

# Import pipelines modules
from src.pipeline.data_processor import DataProcessor

# Standard libraries
import os
import ast
import time
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# ML & NLP
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

In [44]:
processor = DataProcessor()
jobs, x_train, y_train, x_test = processor.load_data()

print(f"Training sessions: {len(x_train)}")
print(f"Training targets: {len(y_train)}")
print(f"Test sessions: {len(x_test)}")
print(f"Job listings: {len(jobs)}")

Training sessions: 15882
Training targets: 15882
Test sessions: 1819
Job listings: 21917


## **Data Processing**

In [45]:
def extract_job_fields(text):
    """
    Extrai os campos principais de uma descrição de job e processa as listas estruturadas.
    Combina extração de texto e parsing de campos como SKILLS, TASKS, LANGUAGES.
    
    Returns:
        dict com os campos processados: TITLE, SUMMARY, DESCRIPTION, 
        HARD_SKILLS, SOFT_SKILLS, LANGUAGES, CERTIFICATIONS, TASKS, COURSES
    """
    # Campos base que buscamos no texto
    MAIN_FIELDS = ['TITLE', 'SUMMARY', 'LANGUAGES', 'CERTIFICATIONS', 'SKILLS', 'COURSES', 'TASKS']
    
    # Inicializa resultado
    # Campos de lista iniciam como lista para consistência
    result = {field: [] if field in ['LANGUAGES', 'CERTIFICATIONS', 'TASKS'] else '' for field in MAIN_FIELDS}
    result['DESCRIPTION'] = ''
    result['HARD_SKILLS'] = []
    result['SOFT_SKILLS'] = []
    
    lines = text.split('\n')
    current_field = None
    buffer = []
    i = 0
    
    def parse_list_content(text_content, extract_names_only=False):
        """Helper para parsear conteúdo de listas formatadas como string de dicts"""
        if not text_content: return []
        items = []
        # Divide por linhas e busca patterns de dicionário/lista (linhas começando com hífens)
        for line in text_content.split('\n'):
            line = line.strip()
            if line.startswith('-'):
                try:
                    # Remove o hífen e avalia
                    val = ast.literal_eval(line[1:].strip())
                    if isinstance(val, dict):
                        if extract_names_only:
                            items.append(val.get('name', ''))
                        else:
                            items.append(val)
                except (ValueError, SyntaxError):
                    continue
        return items
    
    def save_buffer():
        """Salva e processa o buffer no campo atual"""
        if not current_field or not buffer:
            return
            
        content = '\n'.join(buffer).strip()
        
        # Tratamento especial e parsing por campo
        if current_field == 'TITLE' and len(buffer) > 1:
            result['TITLE'] = buffer[0].strip()
            result['SUMMARY'] = '\n'.join(buffer[1:]).strip()
            
        elif current_field == 'SKILLS':
            # Parse de skills e separação
            skills_data = parse_list_content(content, extract_names_only=False)
            result['HARD_SKILLS'] = [s['name'] for s in skills_data if s.get('type') == 'hard']
            result['SOFT_SKILLS'] = [s['name'] for s in skills_data if s.get('type') == 'soft']
            
        elif current_field in ['TASKS', 'LANGUAGES', 'CERTIFICATIONS']:
            result[current_field] = parse_list_content(content, extract_names_only=True)
            
        elif current_field == 'DESCRIPTION':
            if result['DESCRIPTION']:
                result['DESCRIPTION'] += '\n' + content
            else:
                result['DESCRIPTION'] = content
                
        else:
            # Outros campos permanecem como texto (Ex: COURSES se não for lista estruturada)
            result[current_field] = content

    while i < len(lines):
        line = lines[i].strip()
        
        # Detecta campos principais
        if line in MAIN_FIELDS:
            save_buffer()
            buffer = []
            current_field = line
            i += 1
            continue
        
        # Trata SECTION: seguido de DESCRIPTION:
        if line == 'SECTION:':
            save_buffer()
            buffer = []
            current_field = None
            
            # Verifica se a próxima linha é DESCRIPTION:
            if i + 1 < len(lines):
                next_line = lines[i + 1].strip()
                if next_line.startswith('DESCRIPTION:'):
                    desc_content = next_line[12:].strip()
                    desc_lines = [desc_content] if desc_content else []
                    
                    # Coleta todas as linhas até o próximo campo
                    i += 2
                    while i < len(lines):
                        line = lines[i].strip()
                        if line in MAIN_FIELDS + ['SECTION:']:
                            i -= 1  # Volta para processar no próximo loop
                            break
                        if line and not line.startswith('TITLE:'):
                            desc_lines.append(line)
                        i += 1
                    
                    # Adiciona ao resultado
                    if desc_lines:
                        desc_text = '\n'.join(desc_lines)
                        if result['DESCRIPTION']:
                            result['DESCRIPTION'] += '\n' + desc_text
                        else:
                            result['DESCRIPTION'] = desc_text
            i += 1
            continue
        
        # Adiciona linha ao buffer do campo atual
        if current_field and line and not line.startswith(('TITLE:', 'DESCRIPTION:')):
            buffer.append(line)
        
        i += 1
    
    # Salva o último buffer
    save_buffer()
    
    # Remove chave temporária SKILLS se existir, pois já separamos
    if 'SKILLS' in result:
        del result['SKILLS']
    
    return result


# Extrair informações de todos os jobs (agora já parseado)
jobs_extracted = {job_id: extract_job_fields(job_text) for job_id, job_text in jobs.items()}

# Criar DataFrame para visualização
df_jobs = pd.DataFrame.from_dict(jobs_extracted, orient='index')
df_jobs.index.name = 'job_id'
df_jobs = df_jobs.reset_index()

# Converter job_id para int (movido da célula seguinte)
df_jobs['job_id'] = df_jobs['job_id'].astype(int)

print(f"Total jobs: {len(df_jobs)}")
df_jobs.head()

Total jobs: 21917


,job_id,TITLE,SUMMARY,LANGUAGES,CERTIFICATIONS,COURSES,TASKS,DESCRIPTION,HARD_SKILLS,SOFT_SKILLS
0,0,QA Intégration / Data Analyst - SalesForces S...,Responsabilités :\nAssurer la qualité des do...,[],[],,"[assurer la qualité des données intégrées,...",Responsabilités :\nAssurer la qualité des do...,[Salesforce Sales Cloud],[]
1,1,Ingénieur Système,Nous recherchons un Ingénieur Système pour n...,[],[],,"[administrer des serveurs middleware, administ...",Nous recherchons un Ingénieur Système pour n...,"[Administration linux, Administration Windows,...",[]
2,2,Testeur QA Automatisation Cypress,Vous avez au moins une première expérience s...,[],[],,"[approche axée sur les solutions, assister le...",Vous avez au moins une première expérience s...,"[Cypress, Jenkins, Jest, cloud, cucumber, cypr...",[]
3,3,Ingénieur support N3 IP - PARIS,Dans le cadre de cette mission :\nVous garanti...,[],[],,[apporter un support technique sur le fonction...,Dans le cadre de cette mission :\nVous garanti...,"[Adressage IP, BGP (Border Gateway Protocol), ...","[Communication, Adaptabilité, Curiosité, Ges..."
4,4,Business Analyst MOA FRONT,Nous recherchons un (e) consultant(e) ayant un...,[],[],,[accompagnement de la maitrise d’œuvre sur la ...,Nous recherchons un (e) consultant(e) ayant un...,"[autonomie, Relation client, crm, moa, pve, pvf]","[Gestion de projets, Gestion des Risques, auto..."


In [46]:
# Criar colunas separadas para jobs viewed e jobs applied
# Primeiro converter as strings para listas
x_train['job_ids_list'] = x_train['job_ids'].apply(ast.literal_eval)
x_train['actions_list'] = x_train['actions'].apply(ast.literal_eval)

# Agora criar as colunas de jobs viewed e applied
x_train['jobs_viewed'] = x_train.apply(
    lambda row: [job_id for job_id, action in zip(row['job_ids_list'], row['actions_list']) if action == 'view'],
    axis=1
)

x_train['jobs_applied'] = x_train.apply(
    lambda row: [job_id for job_id, action in zip(row['job_ids_list'], row['actions_list']) if action == 'apply'],
    axis=1
)

x_train.drop(['job_ids', 'actions'], axis=1, inplace=True)
x_train.head()

,session_id,job_ids_list,actions_list,jobs_viewed,jobs_applied
0,0,"[305, 299, 300, 290, 282, 274, 264, 261]","[view, view, view, view, view, view, view, view]","[305, 299, 300, 290, 282, 274, 264, 261]",[]
1,1,"[84, 257, 252, 250]","[view, view, view, view]","[84, 257, 252, 250]",[]
2,2,"[241, 237, 221, 309, 310, 306, 301]","[view, view, apply, apply, apply, apply, apply]","[241, 237]","[221, 309, 310, 306, 301]"
3,3,"[303, 297, 296, 298, 294, 295, 292, 293]","[apply, apply, apply, apply, apply, apply, app...",[],"[303, 297, 296, 298, 294, 295, 292, 293]"
4,4,"[171, 291, 289, 166, 288, 155]","[apply, apply, apply, apply, apply, apply]",[],"[171, 291, 289, 166, 288, 155]"


In [47]:
df_jobs_aux = pd.DataFrame.from_dict(jobs, orient='index')
df_jobs_aux

,0
0,TITLE\nQA Intégration / Data Analyst - SalesF...
1,TITLE\nIngénieur Système\n\nSUMMARY\nNous re...
2,TITLE\nTesteur QA Automatisation Cypress\n\nSU...
3,TITLE\nIngénieur support N3 IP - PARIS \n\nSU...
4,TITLE\nBusiness Analyst MOA FRONT\n\nSUMMARY\n...
...,...
27364,TITLE\nIncident Manager e-commerce\n\nSUMMARY\...
27365,TITLE\nConsultant Azure Security\n\nSUMMARY\nK...
27366,TITLE\nChef de projet Supply Chain\n\nSUMMARY\...
27367,TITLE\nPO Infrastructure\n\nSUMMARY\nUn Produc...


In [48]:
text = df_jobs_aux[0].tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')
job_embeddings = model.encode(text, show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/685 [00:00<?, ?it/s]

In [54]:
df_jobs['EMBEDDINGS'] = list(job_embeddings)
df_jobs_embeddings = df_jobs[['job_id', 'EMBEDDINGS']]

# output_dir = 'data/job_listing_feature'
# os.makedirs(output_dir, exist_ok=True)

# df_jobs.to_csv(f'{output_dir}/jobs_with_embeddings.csv', index=False)

In [ ]:
df_jobs_embeddings
df_jobs_embeddings.to_csv('data/job_listing_feature/jobs_with_embeddings.csv', index=False)